In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text('sessions',defaultValue='sessions')
sessions=dbutils.widgets.get('sessions')

In [0]:
sessions_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("session_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("station_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("started_at", StringType(), True),
    StructField("ended_at", StringType(), True),
    StructField("duration_min", IntegerType(), True),
    StructField("energy_kwh", StringType(), True),
    StructField("cost_aud", StringType(), True),
    StructField("peak_power_kw", StringType(), True),
    StructField("connector_type", StringType(), True),
    StructField("session_status", StringType(), True),
    StructField("payment_id", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
])

overall_schema = StructType([
    StructField("data", ArrayType(sessions_schema), True),
    StructField("ingestion_time", StringType(), True),
    StructField("batch_no", StringType(), True),
])

In [0]:
df = (
    spark.readStream.
    format("cloudFiles").schema(overall_schema)
    .option("cloudFiles.format", "json")
    .option("cloudFiles.maxFilesPerTrigger", "1")  # test only
    .option("multiLine", "true")
    # .option("rescuedDataColumn", "_rescued_data")
    .load(f"abfss://raw@voltgridstorageaccount.dfs.core.windows.net/{sessions}/")
)

In [0]:
bronze_df=df.select(F.explode('data')\
    .alias('data'),'ingestion_time','batch_no')\
        .select('data.*','ingestion_time','batch_no')

In [0]:
bronze_df=bronze_df.withColumn('created_at', F.col('created_at').try_cast('date'))\
    .withColumn('updated_at', F.col('updated_at').try_cast('date'))\
        .withColumn('ingestion_time',F.col('ingestion_time').try_cast('timestamp'))\
            .withColumn('id',F.col('id').cast('int'))\
                .withColumn('batch_no',F.col('batch_no').cast('int'))\
                    .withColumn('source_path',F.col('_metadata.file_path'))

In [0]:
q=(bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"abfss://controls@voltgridstorageaccount.dfs.core.windows.net/checkpoints/{sessions}/")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(f"voltgrid.bronze.{sessions}"))
q.awaitTermination()